In [1]:
# ============================================================
# Notebook : 05_baselines.ipynb
# Project  : Causal-Guided Multimodal Sensor Fusion for Elderly Risk Detection
# Purpose  : Baseline models evaluated under an IDENTICAL, fair protocol to the
#            proposed model: same household-level Group 5-fold split, the same
#            training-only standardization, and the same class weighting.
#            Baselines: Single-Modal MLP (EM only), Early-Fusion MLP (all
#            sensors), Attention Fusion (EM/SM cross-attention), and a
#            Transformer encoder (timestep-as-token). This corrects the original
#            comparison, in which baselines were reported below chance (AUC<0.5).
# Note     : Anonymous repository for double-blind review.
# ============================================================

from pathlib import Path
import numpy as np, pandas as pd, random
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import f1_score, roc_auc_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(2)
EPOCHS = 25; LR = 1e-3; WD = 1e-4; BATCH = 256

ROOT     = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC_DIR = ROOT / "data" / "processed"; TAB_DIR = ROOT / "results" / "tables"
TAB_DIR.mkdir(parents=True, exist_ok=True)

z = np.load(PROC_DIR / "windows.npz", allow_pickle=True)
X = z["X"].astype(np.float32); y = z["y"].astype(np.int64)
sample_fold = z["sample_fold"].astype(int)
FEAT = list(z["feat_cols"]); N = len(FEAT); WINDOW = int(z["window"])
EM_IDX = list(z["em_idx"]); SM_IDX = list(z["sm_idx"])
N_FOLDS = int(sample_fold.max()) + 1

def standardize(Xtr_raw, Xall_raw):
    mu = Xtr_raw.mean(axis=(0, 2), keepdims=True)
    sd = Xtr_raw.std(axis=(0, 2), keepdims=True); sd[sd == 0] = 1.0
    return (Xall_raw - mu) / sd

# ── Baseline architectures ───────────────────────────────────
class MLP(nn.Module):
    def __init__(self, d, hid=(256, 128)):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, hid[0]), nn.ReLU(), nn.Dropout(0.3),
                                 nn.Linear(hid[0], hid[1]), nn.ReLU(), nn.Dropout(0.3),
                                 nn.Linear(hid[1], 2))
    def forward(self, x): return self.net(x)

class AttnFusion(nn.Module):
    def __init__(self, em_dim, sm_dim, hid=64):
        super().__init__()
        self.em = nn.Sequential(nn.Linear(em_dim, hid), nn.ReLU())
        self.sm = nn.Sequential(nn.Linear(sm_dim, hid), nn.ReLU())
        self.att = nn.MultiheadAttention(hid, 4, batch_first=True, dropout=0.1)
        self.clf = nn.Sequential(nn.Linear(hid, 32), nn.ReLU(), nn.Dropout(0.3), nn.Linear(32, 2))
    def forward(self, em_x, sm_x):
        e = self.em(em_x); s = self.sm(sm_x)
        tok = torch.stack([e, s], dim=1)
        out, _ = self.att(tok, tok, tok)
        return self.clf(out.mean(1))

class TransformerFusion(nn.Module):
    def __init__(self, d_model=N, nhead=4, layers=2, ff=128, dp=0.1):
        super().__init__()
        enc = nn.TransformerEncoderLayer(d_model, nhead, ff, dp, batch_first=True)
        self.enc = nn.TransformerEncoder(enc, layers)
        self.clf = nn.Sequential(nn.Linear(d_model, 32), nn.ReLU(), nn.Dropout(dp), nn.Linear(32, 2))
    def forward(self, x_seq):      # (B, W, N)
        return self.clf(self.enc(x_seq).mean(1))

def make_dl(arr, idx, shuffle):
    tens = [torch.tensor(a[idx]) for a in arr]
    return DataLoader(TensorDataset(*tens, torch.tensor(y[idx])), batch_size=BATCH, shuffle=shuffle)

def run(kind, Xs, idx_tr, idx_va, idx_te, cw):
    # build per-kind input tensors
    flat = Xs.reshape(Xs.shape[0], -1)                       # (n, N*W)
    em_flat = Xs[:, EM_IDX, :].reshape(Xs.shape[0], -1)      # (n, |EM|*W)
    sm_flat = Xs[:, SM_IDX, :].reshape(Xs.shape[0], -1)
    seq = np.transpose(Xs, (0, 2, 1))                        # (n, W, N)
    if kind == "Single-Modal MLP (EM)":
        arr = [em_flat]; model = MLP(len(EM_IDX) * WINDOW)
    elif kind == "Early Fusion MLP":
        arr = [flat]; model = MLP(N * WINDOW)
    elif kind == "Attention Fusion":
        arr = [em_flat, sm_flat]; model = AttnFusion(len(EM_IDX) * WINDOW, len(SM_IDX) * WINDOW)
    else:  # Transformer Fusion
        arr = [seq]; model = TransformerFusion()
    torch.manual_seed(SEED)
    tr = make_dl(arr, idx_tr, True); va = make_dl(arr, idx_va, False); te = make_dl(arr, idx_te, False)
    crit = nn.CrossEntropyLoss(weight=cw); opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    best = -1; best_state = None
    for ep in range(EPOCHS):
        model.train()
        for batch in tr:
            *xs, yb = batch; loss = crit(model(*xs), yb)
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        sch.step(); model.eval(); P = []; T = []
        with torch.no_grad():
            for batch in va:
                *xs, yb = batch; P.extend(model(*xs).argmax(1).numpy()); T.extend(yb.numpy())
        vf = f1_score(T, P, average="macro", zero_division=0)
        if vf > best: best = vf; best_state = {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state); model.eval()
    P = []; T = []; PR = []
    with torch.no_grad():
        for batch in te:
            *xs, yb = batch; lo = model(*xs)
            P.extend(lo.argmax(1).numpy()); T.extend(yb.numpy()); PR.extend(F.softmax(lo, 1)[:, 1].numpy())
    return (f1_score(T, P, average="macro", zero_division=0),
            f1_score(T, P, average="binary", zero_division=0), roc_auc_score(T, PR))

BASELINES = ["Single-Modal MLP (EM)", "Early Fusion MLP", "Attention Fusion", "Transformer Fusion"]
rows = []
for f in range(N_FOLDS):
    va = (f + 1) % N_FOLDS
    idx_te = np.where(sample_fold == f)[0]
    idx_va = np.where(sample_fold == va)[0]
    idx_tr = np.where((sample_fold != f) & (sample_fold != va))[0]
    Xs = standardize(X[idx_tr], X)
    cw = torch.tensor([1.0, (y[idx_tr] == 0).sum() / max((y[idx_tr] == 1).sum(), 1)], dtype=torch.float)
    for kind in BASELINES:
        mf, bf, auc = run(kind, Xs, idx_tr, idx_va, idx_te, cw)
        rows.append({"model": kind, "fold": f, "macro_f1": round(mf, 4),
                     "binary_f1": round(bf, 4), "auc_roc": round(auc, 4)})
        print(f"fold {f} | {kind:<24} macroF1={mf:.3f} AUC={auc:.3f}")

res = pd.DataFrame(rows); res.to_csv(TAB_DIR / "cv_baselines.csv", index=False)
print("\nBaseline summary (mean +/- std over folds):")
for kind in BASELINES:
    sub = res[res.model == kind]
    print(f"  {kind:<24} AUC {sub.auc_roc.mean():.3f}+/-{sub.auc_roc.std():.3f} | "
          f"macroF1 {sub.macro_f1.mean():.3f}+/-{sub.macro_f1.std():.3f}")
print("Saved -> results/tables/cv_baselines.csv")
print("05_baselines complete.")


fold 0 | Single-Modal MLP (EM)    macroF1=0.480 AUC=0.585


fold 0 | Early Fusion MLP         macroF1=0.573 AUC=0.531


fold 0 | Attention Fusion         macroF1=0.575 AUC=0.562


fold 0 | Transformer Fusion       macroF1=0.546 AUC=0.517


fold 1 | Single-Modal MLP (EM)    macroF1=0.675 AUC=0.745


...

fold 3 | Transformer Fusion       macroF1=0.692 AUC=0.879


fold 4 | Single-Modal MLP (EM)    macroF1=0.496 AUC=0.535


fold 4 | Early Fusion MLP         macroF1=0.606 AUC=0.634


fold 4 | Attention Fusion         macroF1=0.646 AUC=0.675


fold 4 | Transformer Fusion       macroF1=0.669 AUC=0.708

Baseline summary (mean +/- std over folds):
  Single-Modal MLP (EM)    AUC 0.678+/-0.110 | macroF1 0.578+/-0.089
  Early Fusion MLP         AUC 0.738+/-0.149 | macroF1 0.664+/-0.069
  Attention Fusion         AUC 0.751+/-0.131 | macroF1 0.675+/-0.066
  Transformer Fusion       AUC 0.754+/-0.147 | macroF1 0.671+/-0.074
Saved -> results/tables/cv_baselines.csv
05_baselines complete.
